In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

dataset_path = "/home/thor/SpatialVLA/smoke_test/data/bridge_dataset-train.tfrecord-00000-of-01024"

def show_full_contents(path, max_episodes=50):
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return

    print(f"Reading dataset from: {path}")
    raw_dataset = tf.data.TFRecordDataset(path)

    iterator = iter(raw_dataset)
    episode_num = 0

    while episode_num < max_episodes:
        try:
            raw_record = next(iterator)
            episode_num += 1
            print(f"\n{'='*70}")
            print(f"EPISODE {episode_num}")
            print(f"{'='*70}")

            example = tf.train.Example()
            example.ParseFromString(raw_record.numpy())
            features = example.features.feature

            # ========== EPISODE METADATA ==========
            print("\n--- EPISODE METADATA ---")

            # Episode ID
            if 'episode_metadata/episode_id' in features:
                feat_id = features['episode_metadata/episode_id']
                if feat_id.int64_list.value:
                    ep_id = feat_id.int64_list.value[0]
                    print(f"Episode ID: {ep_id}")
                elif feat_id.bytes_list.value:
                    print(f"Episode ID: {feat_id.bytes_list.value[0].decode('utf-8')}")

            # File Path
            if 'episode_metadata/file_path' in features:
                file_path = features['episode_metadata/file_path'].bytes_list.value[0].decode('utf-8')
                print(f"File Path: {file_path}")

            # Image availability flags
            has_images = {}
            for i in range(4):
                key = f'episode_metadata/has_image_{i}'
                if key in features:
                    has_images[i] = features[key].int64_list.value[0] if features[key].int64_list.value else 0
            if has_images:
                available = [f'image_{i}' for i, v in has_images.items() if v]
                print(f"Available Images: {available}")

            # Language availability
            if 'episode_metadata/has_language' in features:
                has_lang = features['episode_metadata/has_language'].int64_list.value[0] if features['episode_metadata/has_language'].int64_list.value else 0
                print(f"Has Language Instruction: {bool(has_lang)}")

            # ========== EPISODE-LEVEL DATA ==========
            print("\n--- EPISODE DATA ---")

            # Language Instruction
            if 'steps/language_instruction' in features:
                lang_bytes = features['steps/language_instruction'].bytes_list.value
                if lang_bytes:
                    instruction = lang_bytes[0].decode('utf-8')
                    print(f"Language Instruction: {instruction}")

            # Count steps from action array
            if 'steps/action' in features:
                action_floats = features['steps/action'].float_list.value
                num_steps = len(action_floats) // 7  # 7D actions
                print(f"Number of Steps: {num_steps}")

            # Show first step action as example
            if 'steps/action' in features and 'steps/observation/state' in features:
                action_floats = features['steps/action'].float_list.value
                state_floats = features['steps/observation/state'].float_list.value
                if len(action_floats) >= 7:
                    first_action = action_floats[:7]
                    print(f"First Step Action (7D): XYZ=[{first_action[0]:.4f}, {first_action[1]:.4f}, {first_action[2]:.4f}], "
                          f"RPY=[{first_action[3]:.4f}, {first_action[4]:.4f}, {first_action[5]:.4f}], "
                          f"Gripper={first_action[6]:.4f}")

            # ========== SHOW FIRST FRAME IMAGES ==========
            print("\n--- FIRST FRAME IMAGES ---")
            image_keys = [f'steps/observation/image_{i}' for i in range(4)]
            images_to_show = []

            for k in image_keys:
                if k in features and features[k].bytes_list.value:
                    try:
                        img_bytes = features[k].bytes_list.value[0]  # First step image
                        img = tf.image.decode_image(img_bytes).numpy()
                        images_to_show.append((k, img))
                    except:
                        pass

            if images_to_show:
                cols = len(images_to_show)
                plt.figure(figsize=(4 * cols, 4))
                for idx, (name, img) in enumerate(images_to_show):
                    plt.subplot(1, cols, idx+1)
                    plt.imshow(img)
                    plt.title(f"{name}\n{img.shape}")
                    plt.axis('off')
                plt.show()
            else:
                print("No images available")

        except StopIteration:
            print("\n--- End of File (Complete) ---")
            break
        except Exception as e:
            print(f"\nStopped due to error: {e}")
            import traceback
            traceback.print_exc()
            break

show_full_contents(dataset_path)

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

dataset_path = "/home/thor/SpatialVLA/smoke_test/data/bridge_dataset-train.tfrecord-00000-of-01024"

if not os.path.exists(dataset_path):
    print(f"File not found: {dataset_path}")
else:
    raw_dataset = tf.data.TFRecordDataset(dataset_path)

    # Take the first episode (Episode 1 / Index 0)
    for raw_record in raw_dataset.take(1):
        example = tf.train.Example()
        example.ParseFromString(raw_record.numpy())
        features = example.features.feature

        # --- Metadata ---
        instruction = "N/A"
        if 'steps/language_instruction' in features:
             vals = features['steps/language_instruction'].bytes_list.value
             if vals: instruction = vals[0].decode('utf-8')

        ep_id = "N/A"
        if 'episode_metadata/episode_id' in features:
             vals = features['episode_metadata/episode_id'].bytes_list.value
             if vals: ep_id = vals[0].decode('utf-8')

        print(f"Episode ID: {ep_id}")
        print(f"Instruction: {instruction}")

        # --- Data Extraction ---
        # 1. Images (Primary)
        images = []
        if 'steps/observation/image_0' in features:
            img_bytes_list = features['steps/observation/image_0'].bytes_list.value
            for b in img_bytes_list:
                img = tf.image.decode_image(b).numpy()
                images.append(img)

        # 2. Actions
        actions = []
        if 'steps/action' in features:
            action_floats = features['steps/action'].float_list.value
            if len(images) > 0:
                 dim = len(action_floats) // len(images)
                 actions = np.array(action_floats).reshape(len(images), dim)

        print(f"Total Steps: {len(images)}")

        # --- Plotting ---
        # Show steps in a grid
        cols = 5
        rows = (len(images) + cols - 1) // cols

        if len(images) > 0:
            plt.figure(figsize=(20, 4 * rows))
            for i, img in enumerate(images):
                plt.subplot(rows, cols, i + 1)
                plt.imshow(img)

                title = f"Step {i+1}"
                if len(actions) > i:
                    # Show X, Y, Z, Gripper
                    act = actions[i]
                    # Assuming last dim is gripper
                    grip = "Open" if act[-1] > 0.5 else "Closed"
                    title += f"\nXYZ: [{act[0]:.2f}, {act[1]:.2f}, {act[2]:.2f}]\nGrip: {grip}"

                plt.title(title, fontsize=8)
                plt.axis('off')
            plt.tight_layout()
            plt.show()
        else:
            print("No images found in this episode.")

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

dataset_path = "/home/thor/SpatialVLA/smoke_test/data/bridge_dataset-train.tfrecord-00000-of-01024"

if not os.path.exists(dataset_path):
    print(f"File not found: {dataset_path}")
else:
    raw_dataset = tf.data.TFRecordDataset(dataset_path)

    for raw_record in raw_dataset.take(1):
        example = tf.train.Example()
        example.ParseFromString(raw_record.numpy())
        features = example.features.feature

        # --- Episode Metadata ---
        print("="*70)
        print("EPISODE METADATA")
        print("="*70)

        ep_id = "N/A"
        if 'episode_metadata/episode_id' in features:
            feat_id = features['episode_metadata/episode_id']
            if feat_id.int64_list.value:
                ep_id = str(feat_id.int64_list.value[0])
            elif feat_id.bytes_list.value:
                ep_id = feat_id.bytes_list.value[0].decode('utf-8')

        file_path = "N/A"
        if 'episode_metadata/file_path' in features:
            vals = features['episode_metadata/file_path'].bytes_list.value
            if vals: file_path = vals[0].decode('utf-8')

        instruction = "N/A"
        if 'steps/language_instruction' in features:
            vals = features['steps/language_instruction'].bytes_list.value
            if vals: instruction = vals[0].decode('utf-8')

        print(f"Episode ID: {ep_id}")
        print(f"File Path: {file_path}")
        print(f"Instruction: {instruction}")

        # Check which images are available
        has_images = {}
        for i in range(4):
            key = f'episode_metadata/has_image_{i}'
            if key in features:
                has_images[i] = features[key].int64_list.value[0] if features[key].int64_list.value else 0

        print(f"Available Images: {[f'image_{i}' for i, v in has_images.items() if v]}")

        # --- Extract Step Data ---
        # Get number of steps from any sequence feature
        num_steps = 0
        if 'steps/action' in features:
            action_floats = features['steps/action'].float_list.value
            # Actions are 7D: [x, y, z, roll, pitch, yaw, gripper]
            num_steps = len(action_floats) // 7

        print(f"\nTotal Steps: {num_steps}")
        print("="*70)

        # Extract all step data
        images_dict = {}
        for i in range(4):
            key = f'steps/observation/image_{i}'
            if key in features:
                img_bytes_list = features[key].bytes_list.value
                images_dict[i] = []
                for b in img_bytes_list:
                    try:
                        img = tf.image.decode_image(b).numpy()
                        images_dict[i].append(img)
                    except:
                        images_dict[i].append(None)

        actions = []
        if 'steps/action' in features:
            action_floats = features['steps/action'].float_list.value
            actions = np.array(action_floats).reshape(num_steps, 7)

        states = []
        if 'steps/observation/state' in features:
            state_floats = features['steps/observation/state'].float_list.value
            state_dim = len(state_floats) // num_steps
            states = np.array(state_floats).reshape(num_steps, state_dim)

        is_first = []
        if 'steps/is_first' in features:
            is_first = features['steps/is_first'].int64_list.value

        is_last = []
        if 'steps/is_last' in features:
            is_last = features['steps/is_last'].int64_list.value

        is_terminal = []
        if 'steps/is_terminal' in features:
            is_terminal = features['steps/is_terminal'].int64_list.value

        rewards = []
        if 'steps/reward' in features:
            rewards = features['steps/reward'].float_list.value

        discounts = []
        if 'steps/discount' in features:
            discounts = features['steps/discount'].float_list.value

        # --- Display Each Step ---
        for step_idx in range(num_steps):
            print(f"\n{'='*70}")
            print(f"STEP {step_idx + 1}/{num_steps}")
            print(f"{'='*70}")

            # Step metadata
            print(f"is_first: {is_first[step_idx] if step_idx < len(is_first) else 'N/A'}")
            print(f"is_last: {is_last[step_idx] if step_idx < len(is_last) else 'N/A'}")
            print(f"is_terminal: {is_terminal[step_idx] if step_idx < len(is_terminal) else 'N/A'}")
            print(f"reward: {rewards[step_idx] if step_idx < len(rewards) else 'N/A'}")
            print(f"discount: {discounts[step_idx] if step_idx < len(discounts) else 'N/A'}")

            # Actions
            if step_idx < len(actions):
                act = actions[step_idx]
                print(f"\nAction (7D):")
                print(f"  Position XYZ: [{act[0]:.4f}, {act[1]:.4f}, {act[2]:.4f}]")
                print(f"  Rotation RPY: [{act[3]:.4f}, {act[4]:.4f}, {act[5]:.4f}]")
                print(f"  Gripper: {act[6]:.4f} ({'Open' if act[6] > 0.5 else 'Closed'})")

            # State
            if step_idx < len(states):
                state = states[step_idx]
                print(f"\nState ({len(state)}D): {state[:10]}..." if len(state) > 10 else f"\nState ({len(state)}D): {state}")

            # Images for this step
            images_to_show = []
            for img_idx in range(4):
                if img_idx in images_dict and step_idx < len(images_dict[img_idx]):
                    img = images_dict[img_idx][step_idx]
                    if img is not None:
                        images_to_show.append((f"image_{img_idx}", img))

            if images_to_show:
                cols = len(images_to_show)
                plt.figure(figsize=(4 * cols, 4))
                for idx, (name, img) in enumerate(images_to_show):
                    plt.subplot(1, cols, idx + 1)
                    plt.imshow(img)
                    plt.title(f"Step {step_idx+1}: {name}\n{img.shape}")
                    plt.axis('off')
                plt.tight_layout()
                plt.show()
            else:
                print("\nNo images available for this step.")